In [14]:
import numpy as np 
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import KFold

In this data set there is a group refrenced but no used in the benchmark so this is a place to start to see if they are needed

In [15]:
X_train = pd.read_csv('../data/X_train.csv',index_col='ROW_ID')

y_train = pd.read_csv('../data/y_train.csv',index_col='ROW_ID')
total_data = pd.concat([X_train,y_train],axis=1)
print(total_data['target'])

ROW_ID
0         0.009210
1         0.000405
2         0.005967
3        -0.000542
4        -0.002579
            ...   
527068   -0.001769
527069    0.005640
527070    0.004016
527071    0.001617
527072    0.000556
Name: target, Length: 527073, dtype: float64


In [16]:
number_of_groups = total_data['GROUP'].nunique()
print(f"Number of groups: {number_of_groups}")

Number of groups: 4


In [17]:
total_target_signs = total_data['target'].apply(np.sign)
total_positive_count_percent = (total_target_signs == 1).sum() / len(total_data) * 100
total_negative_count_percent = (total_target_signs == -1).sum() / len(total_data) * 100
print(f"Total: Positive count percent = {total_positive_count_percent}, Negative count percent = {total_negative_count_percent}")
for group in range(1, number_of_groups + 1):
    group_data = total_data[total_data['GROUP'] == group]
    target_signs= group_data['target'].apply(np.sign)
    positive_count_percent = (target_signs == 1).sum() / len(group_data) * 100
    negative_count_percent = (target_signs == -1).sum() / len(group_data) * 100
    print(f"Group {group}: Positive count percent = {positive_count_percent}, Negative count percent = {negative_count_percent}")

Total: Positive count percent = 50.71840143585423, Negative count percent = 49.28159856414576
Group 1: Positive count percent = 50.051911750024956, Negative count percent = 49.948088249975044
Group 2: Positive count percent = 50.97829953753112, Negative count percent = 49.02170046246887
Group 3: Positive count percent = 50.99193075725373, Negative count percent = 49.00806924274627
Group 4: Positive count percent = 50.60051031187531, Negative count percent = 49.39948968812469


Now looking at this result i have not previously looked at just the positive negtive split

In [19]:
train_dates = X_train['TS'].unique()
n_splits = 8
scores_always_long = []

splits = KFold(n_splits=n_splits, random_state=0, shuffle=True).split(train_dates)
for i, (local_train_dates_ids, local_test_dates_ids) in enumerate(splits):
    local_test_dates = train_dates[local_test_dates_ids]
    local_test_ids = X_train['TS'].isin(local_test_dates)

    y_local_test = y_train.loc[local_test_ids, 'target']

    always_long_preds = np.ones(len(y_local_test))
    score = accuracy_score((y_local_test > 0).astype(int), always_long_preds)
    scores_always_long.append(score)
    print(f"Fold {i+1} - Always-long accuracy: {score * 100:.2f}%")

mean = np.mean(scores_always_long) * 100
std = np.std(scores_always_long) * 100
print(f"\nAlways-long accuracy: {mean:.2f}% (+- {std:.2f})")

Fold 1 - Always-long accuracy: 50.99%
Fold 2 - Always-long accuracy: 50.97%
Fold 3 - Always-long accuracy: 51.53%
Fold 4 - Always-long accuracy: 50.67%
Fold 5 - Always-long accuracy: 50.56%
Fold 6 - Always-long accuracy: 50.66%
Fold 7 - Always-long accuracy: 50.37%
Fold 8 - Always-long accuracy: 50.00%

Always-long accuracy: 50.72% (+- 0.43)


Market go up big breakthrough now lets see the groups changing over time to see if they give any insight

In [20]:
unique_ts = total_data['TS'].unique()
n_chunks = 4
ts_chunk_map = {ts: i % n_chunks for i, ts in enumerate(unique_ts)}
total_data['TS_CHUNK'] = total_data['TS'].map(ts_chunk_map)

result = (
    total_data
    .assign(target_sign=np.sign(total_data['target']))
    .groupby(['GROUP', 'TS_CHUNK'])['target_sign']
    .apply(lambda s: (s == 1).mean() * 100)
    .unstack('TS_CHUNK')
)

print(result)
print("\nStd across chunks per group (lower = more stable):")
print(result.std(axis=1))

TS_CHUNK          0          1          2          3
GROUP                                               
1         49.765277  50.273810  50.460048  49.713501
2         50.618775  50.744994  51.143512  51.398581
3         51.538864  50.488099  51.177082  50.766791
4         50.912719  49.186644  52.152987  50.116279

Std across chunks per group (lower = more stable):
GROUP
1    0.370805
2    0.359448
3    0.461140
4    1.257097
dtype: float64


4 has a large standard deviation meaning the constant positive trend is harder to trust in this data set as we see the low negtive value in the chink also chunk 1 does have some negtive biasis have a positive bias hardcode may be tricky

In [21]:
# sign-based autocorrelation: does yesterday's direction predict tomorrow's?
ret_cols = [f'RET_{i}' for i in range(1, 21)]

target_sign = np.sign(total_data['target'])

for col in ret_cols:
    ret_sign = np.sign(total_data[col])
    # simple match rate: how often does lag-i's sign match tomorrow's sign?
    match_rate = (ret_sign == target_sign).mean() * 100
    print(f"{col}: sign-match rate with target = {match_rate:.2f}%")

RET_1: sign-match rate with target = 51.89%
RET_2: sign-match rate with target = 49.75%
RET_3: sign-match rate with target = 49.95%
RET_4: sign-match rate with target = 49.73%
RET_5: sign-match rate with target = 49.96%
RET_6: sign-match rate with target = 50.16%
RET_7: sign-match rate with target = 50.40%
RET_8: sign-match rate with target = 50.45%
RET_9: sign-match rate with target = 50.43%
RET_10: sign-match rate with target = 50.21%
RET_11: sign-match rate with target = 50.28%
RET_12: sign-match rate with target = 50.38%
RET_13: sign-match rate with target = 50.18%
RET_14: sign-match rate with target = 50.25%
RET_15: sign-match rate with target = 50.12%
RET_16: sign-match rate with target = 50.17%
RET_17: sign-match rate with target = 49.94%
RET_18: sign-match rate with target = 50.23%
RET_19: sign-match rate with target = 50.29%
RET_20: sign-match rate with target = 50.30%


In [22]:
for col in ret_cols:
    corr = total_data[col].corr(total_data['target'])
    print(f"{col}: Pearson corr with target = {corr:.4f}")

RET_1: Pearson corr with target = 0.0849
RET_2: Pearson corr with target = -0.0090
RET_3: Pearson corr with target = -0.0121
RET_4: Pearson corr with target = -0.0243
RET_5: Pearson corr with target = -0.0014
RET_6: Pearson corr with target = 0.0118
RET_7: Pearson corr with target = 0.0212
RET_8: Pearson corr with target = 0.0207
RET_9: Pearson corr with target = 0.0196
RET_10: Pearson corr with target = -0.0020
RET_11: Pearson corr with target = 0.0025
RET_12: Pearson corr with target = -0.0016
RET_13: Pearson corr with target = 0.0048
RET_14: Pearson corr with target = 0.0076
RET_15: Pearson corr with target = 0.0021
RET_16: Pearson corr with target = 0.0103
RET_17: Pearson corr with target = -0.0030
RET_18: Pearson corr with target = -0.0100
RET_19: Pearson corr with target = 0.0044
RET_20: Pearson corr with target = 0.0027


So the strongest signal is yesterdays returns its strong but not masivly worth checking if they alone can perform well 

In [23]:

ret1_sign = np.sign(total_data['RET_1'])
target_sign = np.sign(total_data['target'])

match_rate = (ret1_sign == target_sign).mean() * 100
print(f"RET_1 sign-match rate with target = {match_rate:.2f}%")

# also check overall predicted-positive rate if you naively used sign(RET_1) as the prediction
pred_positive_rate = (ret1_sign == 1).mean() * 100
print(f"sign(RET_1) predicted-positive rate = {pred_positive_rate:.2f}%")

RET_1 sign-match rate with target = 51.89%
sign(RET_1) predicted-positive rate = 51.17%
